<a href="https://colab.research.google.com/github/kunal190399/agentic_ai_task_orchestration/blob/main/agentic_ai_task_orchestration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install all required libraries
# Run this cell once — it may take 1-2 minutes

!pip install langchain langchain-openai langchain-community langgraph openai python-dotenv networkx matplotlib --quiet

print("✅ All libraries installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.7/506.7 kB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.0 which is incompatible.
✅ All libraries installed


In [2]:
import os
import json
import time
import random
import hashlib
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from typing import Dict, List, Any, Optional
from dataclasses import dataclass, field

print("✅ Libraries imported")

# ── What are we building? ────────────────────────────────────────
# A multi-agent AI system that:
# 1. Takes a complex user task as input
# 2. PLANNER AGENT breaks it into subtasks with dependencies
# 3. EXECUTOR AGENTS handle each subtask autonomously
# 4. CRITIC AGENT evaluates quality before returning output
# 5. Full decision trace logs every agent action for explainability
#
# This directly maps onto the PhD's three aims:
# Aim 1: Multi-agent AI architecture for autonomous workflows
# Aim 2: Planning, scheduling, and dynamic task allocation
# Aim 3: Controllable, explainable, deployable agentic AI

✅ Libraries imported


In [3]:
# ── Task and Agent data structures ──────────────────────────────

@dataclass
class Task:
    """
    A single unit of work in the agentic pipeline.
    Tasks form a dependency graph — some must complete before others start.
    This is the fundamental unit of the task orchestration system.
    """
    task_id:      str
    name:         str
    description:  str
    agent_type:   str          # which type of agent should handle this
    dependencies: List[str]    # task_ids that must complete first
    priority:     int          # 1=low, 2=medium, 3=high
    status:       str = "pending"   # pending / running / complete / failed
    result:       Any = None
    start_time:   Optional[float] = None
    end_time:     Optional[float] = None

    @property
    def duration(self):
        if self.start_time and self.end_time:
            return round(self.end_time - self.start_time, 3)
        return None


@dataclass
class AgentAction:
    """
    A single logged action by an agent.
    This is the explainability layer — every decision is recorded.
    Human operators can audit the full reasoning trace.
    """
    timestamp:   str
    agent_name:  str
    action_type: str    # plan / execute / evaluate / schedule / allocate
    task_id:     Optional[str]
    input_data:  str
    output_data: str
    reasoning:   str    # WHY the agent made this decision
    confidence:  float  # 0.0 to 1.0


class DecisionTrace:
    """
    Stores the full audit trail of all agent decisions.
    This is what makes the system controllable and explainable.
    Human operators can review, challenge, or override any decision.
    """
    def __init__(self):
        self.actions: List[AgentAction] = []

    def log(self, agent_name, action_type, task_id,
            input_data, output_data, reasoning, confidence=0.9):
        action = AgentAction(
            timestamp   = datetime.now().strftime("%H:%M:%S.%f")[:-3],
            agent_name  = agent_name,
            action_type = action_type,
            task_id     = task_id,
            input_data  = str(input_data)[:200],
            output_data = str(output_data)[:200],
            reasoning   = reasoning,
            confidence  = confidence
        )
        self.actions.append(action)
        print(f"  [{action.timestamp}] {agent_name} | {action_type}"
              + (f" | Task: {task_id}" if task_id else "")
              + f" | confidence={confidence:.2f}")

    def summary(self):
        print(f"\nTotal logged actions: {len(self.actions)}")
        by_agent = {}
        for a in self.actions:
            by_agent[a.agent_name] = by_agent.get(a.agent_name, 0) + 1
        for agent, count in sorted(by_agent.items()):
            print(f"  {agent}: {count} actions")


# Global trace — shared across all agents
trace = DecisionTrace()
print("✅ Data structures ready")

✅ Data structures ready


In [4]:
# ── AGENT 1: PLANNER AGENT ───────────────────────────────────────
# Receives the user's goal and decomposes it into a task graph.
# Determines dependencies, assigns agent types, sets priorities.
# This is Aim 1 + Aim 2 of the PhD: multi-agent architecture +
# planning and dynamic task allocation.

class PlannerAgent:
    """
    The Planner decomposes complex goals into executable task graphs.
    It determines:
    - What subtasks are needed
    - Which agent type should handle each
    - What dependencies exist between tasks
    - What priority each task has
    """

    def __init__(self):
        self.name = "PlannerAgent"

    def decompose(self, goal: str) -> List[Task]:
        """
        Decompose a high-level goal into a dependency graph of tasks.
        In production this would use an LLM. Here we use rule-based
        decomposition to illustrate the planning logic clearly.
        """
        trace.log(
            agent_name  = self.name,
            action_type = "plan",
            task_id     = None,
            input_data  = goal,
            output_data = "Decomposing into subtask dependency graph",
            reasoning   = "Goal requires multiple specialist agents. "
                         "Decomposing by domain to enable parallel execution "
                         "where dependencies allow.",
            confidence  = 0.92
        )

        # ── Task graph definition ─────────────────────────────────
        # Dependencies form a DAG (directed acyclic graph).
        # Tasks with no dependencies can run immediately (in parallel).
        # Tasks with dependencies wait until all predecessors complete.

        tasks = [
            Task(
                task_id      = "T1",
                name         = "Requirements analysis",
                description  = f"Analyse the goal and identify core requirements: {goal}",
                agent_type   = "AnalysisAgent",
                dependencies = [],          # no dependencies — runs first
                priority     = 3,           # high priority
            ),
            Task(
                task_id      = "T2",
                name         = "Data retrieval",
                description  = "Retrieve relevant data, context, and background information",
                agent_type   = "DataAgent",
                dependencies = [],          # also runs immediately (parallel with T1)
                priority     = 2,
            ),
            Task(
                task_id      = "T3",
                name         = "Core processing",
                description  = "Process and synthesise findings from requirements and data",
                agent_type   = "ProcessingAgent",
                dependencies = ["T1", "T2"],   # waits for both T1 and T2
                priority     = 3,
            ),
            Task(
                task_id      = "T4",
                name         = "Output generation",
                description  = "Generate structured output from processed results",
                agent_type   = "OutputAgent",
                dependencies = ["T3"],
                priority     = 2,
            ),
            Task(
                task_id      = "T5",
                name         = "Quality validation",
                description  = "Validate output quality and completeness",
                agent_type   = "CriticAgent",
                dependencies = ["T4"],
                priority     = 3,
            ),
        ]

        trace.log(
            agent_name  = self.name,
            action_type = "plan",
            task_id     = None,
            input_data  = goal,
            output_data = f"Created {len(tasks)} tasks with dependency graph",
            reasoning   = "T1 and T2 have no dependencies so can run in parallel. "
                         "T3 requires both to complete. T4 and T5 are sequential.",
            confidence  = 0.95
        )

        return tasks


# ── AGENT 2: SCHEDULER AGENT ──────────────────────────────────────
# Determines execution order respecting dependencies.
# Uses topological sort — a classic planning algorithm.
# Identifies which tasks can run in parallel at each stage.
# This is the core of Aim 2: scheduling and dynamic task allocation.

class SchedulerAgent:
    """
    The Scheduler takes the task graph and produces an execution plan.
    It uses topological sorting to find a valid execution order that
    respects all dependencies while maximising parallelism.
    """

    def __init__(self):
        self.name = "SchedulerAgent"

    def schedule(self, tasks: List[Task]) -> List[List[Task]]:
        """
        Returns a list of execution waves.
        Wave 0 = tasks with no dependencies (run in parallel).
        Wave 1 = tasks whose dependencies are all in wave 0.
        And so on.
        """
        task_map = {t.task_id: t for t in tasks}

        trace.log(
            agent_name  = self.name,
            action_type = "schedule",
            task_id     = None,
            input_data  = f"{len(tasks)} tasks to schedule",
            output_data = "Running topological sort on dependency graph",
            reasoning   = "Topological sort guarantees valid execution order. "
                         "Tasks in same wave can run in parallel — "
                         "maximising throughput while respecting dependencies.",
            confidence  = 0.98
        )

        # Build dependency graph and compute in-degrees
        in_degree = {t.task_id: len(t.dependencies) for t in tasks}
        graph     = {t.task_id: [] for t in tasks}
        for t in tasks:
            for dep in t.dependencies:
                graph[dep].append(t.task_id)

        # Kahn's algorithm — produces execution waves
        waves = []
        ready = [tid for tid, deg in in_degree.items() if deg == 0]

        while ready:
            # Sort by priority within each wave (higher priority first)
            wave_tasks = sorted(
                [task_map[tid] for tid in ready],
                key=lambda t: -t.priority
            )
            waves.append(wave_tasks)

            next_ready = []
            for t in wave_tasks:
                for successor in graph[t.task_id]:
                    in_degree[successor] -= 1
                    if in_degree[successor] == 0:
                        next_ready.append(successor)
            ready = next_ready

        for i, wave in enumerate(waves):
            task_names = [t.name for t in wave]
            trace.log(
                agent_name  = self.name,
                action_type = "allocate",
                task_id     = None,
                input_data  = f"Wave {i}",
                output_data = f"Tasks: {task_names}",
                reasoning   = f"These {len(wave)} tasks have all dependencies satisfied "
                             f"and can execute {'in parallel' if len(wave)>1 else 'now'}.",
                confidence  = 0.96
            )

        return waves


# ── AGENT 3: EXECUTOR AGENT ──────────────────────────────────────
# Executes individual tasks autonomously.
# In production, this calls LLM APIs, databases, external tools.
# Here we simulate realistic outputs for each agent type.

class ExecutorAgent:
    """
    Executor agents perform the actual task work.
    Each executor is specialised for a domain (analysis, data, etc).
    They operate autonomously within their scope and log all actions.
    """

    EXECUTOR_TYPES = {
        "AnalysisAgent":    "Analyse requirements and identify key objectives",
        "DataAgent":        "Retrieve and process relevant data and context",
        "ProcessingAgent":  "Synthesise analysis and data into actionable insights",
        "OutputAgent":      "Structure and format results for human consumption",
        "CriticAgent":      "Evaluate quality, completeness, and identify gaps",
    }

    def __init__(self, agent_type: str):
        self.name       = agent_type
        self.capability = self.EXECUTOR_TYPES.get(agent_type, "General execution")

    def execute(self, task: Task, context: Dict) -> str:
        """Execute a task and return a result string."""

        task.status     = "running"
        task.start_time = time.time()

        trace.log(
            agent_name  = self.name,
            action_type = "execute",
            task_id     = task.task_id,
            input_data  = task.description,
            output_data = f"Starting execution: {self.capability}",
            reasoning   = f"This task requires {self.name} capability. "
                         f"All {len(task.dependencies)} dependencies are satisfied.",
            confidence  = 0.88
        )

        # Simulate task execution with realistic outputs
        time.sleep(0.1)   # simulates processing time
        result = self._simulate_execution(task, context)

        task.status   = "complete"
        task.result   = result
        task.end_time = time.time()

        trace.log(
            agent_name  = self.name,
            action_type = "execute",
            task_id     = task.task_id,
            input_data  = task.description,
            output_data = result[:150],
            reasoning   = f"Task completed in {task.duration}s. "
                         f"Result passed to dependent tasks via shared context.",
            confidence  = 0.91
        )

        return result

    def _simulate_execution(self, task: Task, context: Dict) -> str:
        """Simulate realistic agent outputs for each executor type."""
        outputs = {
            "AnalysisAgent": (
                f"REQUIREMENTS ANALYSIS: Identified 3 core objectives. "
                f"Primary goal: {task.description[:60]}. "
                f"Key constraints: time sensitivity (high), accuracy requirement (>95%), "
                f"output format (structured JSON). "
                f"Recommended approach: decomposed execution with validation gate."
            ),
            "DataAgent": (
                f"DATA RETRIEVAL: Located 4 relevant data sources. "
                f"Retrieved 2,847 records. "
                f"Data quality score: 0.91 (1 source had partial coverage). "
                f"Key findings: temporal patterns in Q1-Q3, "
                f"3 anomalous clusters identified for downstream processing."
            ),
            "ProcessingAgent": (
                f"SYNTHESIS COMPLETE: Integrated requirements from T1 and data from T2. "
                f"Identified 2 high-priority action items and 1 risk factor. "
                f"Confidence in synthesis: 0.87. "
                f"Flagged for human review: the anomalous cluster from T2 "
                f"may indicate data quality issue requiring operator attention."
            ),
            "OutputAgent": (
                f"OUTPUT GENERATED: Structured response with 3 sections: "
                f"(1) Executive summary, (2) Detailed findings, (3) Recommendations. "
                f"Format: JSON-compatible structured output. "
                f"Readability score: 0.92. "
                f"All outputs traceable to source data via audit trail."
            ),
            "CriticAgent": (
                f"QUALITY VALIDATION: Reviewed output from T4. "
                f"Completeness: 94% (minor gap in edge case coverage). "
                f"Accuracy: 97% (validated against T2 source data). "
                f"Controllability: PASS (full audit trail present). "
                f"RECOMMENDATION: Accept output. Flag edge case gap for iteration 2."
            ),
        }
        return outputs.get(self.name, f"Task {task.task_id} completed successfully.")

In [5]:
# ── ORCHESTRATOR ─────────────────────────────────────────────────
# The Orchestrator coordinates all agents end-to-end.
# It is the top-level controller that humans interact with.
# This is what makes the system "semi-independent while controllable".

class AgenticOrchestrator:
    """
    The Orchestrator is the human-facing controller of the multi-agent system.
    It:
    1. Receives the user's goal
    2. Invokes the Planner to decompose it
    3. Invokes the Scheduler to order execution
    4. Dispatches Executor agents in the correct wave order
    5. Collects results and passes context between agents
    6. Returns a final result with full audit trail
    """

    def __init__(self):
        self.planner   = PlannerAgent()
        self.scheduler = SchedulerAgent()
        self.executors = {
            name: ExecutorAgent(name)
            for name in ExecutorAgent.EXECUTOR_TYPES
        }

    def run(self, goal: str) -> Dict:
        print("\n" + "="*60)
        print("AGENTIC AI ORCHESTRATOR — STARTING")
        print("="*60)
        print(f"Goal: {goal}\n")

        start = time.time()
        context: Dict[str, Any] = {"goal": goal, "results": {}}

        # Step 1: Plan
        print("--- PHASE 1: PLANNING ---")
        tasks = self.planner.decompose(goal)

        # Step 2: Schedule
        print("\n--- PHASE 2: SCHEDULING ---")
        waves = self.scheduler.schedule(tasks)
        task_map = {t.task_id: t for t in tasks}

        # Step 3: Execute wave by wave
        print("\n--- PHASE 3: EXECUTION ---")
        for wave_num, wave in enumerate(waves):
            parallel = len(wave) > 1
            print(f"\n  Wave {wave_num}: "
                  f"{'Parallel execution' if parallel else 'Sequential execution'} "
                  f"({len(wave)} task{'s' if len(wave)>1 else ''})")

            for task in wave:
                print(f"    Executing: {task.name} [{task.task_id}]")
                executor = self.executors.get(task.agent_type, ExecutorAgent("AnalysisAgent"))
                result   = executor.execute(task, context)
                context["results"][task.task_id] = result

        # Step 4: Compile final output
        elapsed    = round(time.time() - start, 2)
        all_tasks  = list(task_map.values())
        completed  = [t for t in all_tasks if t.status == "complete"]

        final_output = {
            "goal":            goal,
            "status":          "complete" if len(completed) == len(all_tasks) else "partial",
            "tasks_completed": len(completed),
            "tasks_total":     len(all_tasks),
            "execution_waves": len(waves),
            "total_time_s":    elapsed,
            "agent_actions":   len(trace.actions),
            "final_result":    context["results"].get("T5", "See task results"),
            "task_results":    {t.task_id: {
                "name":     t.name,
                "status":   t.status,
                "duration": t.duration,
                "result":   str(t.result)[:200] if t.result else None
            } for t in all_tasks}
        }

        print("\n" + "="*60)
        print("EXECUTION COMPLETE")
        print("="*60)
        print(f"Tasks completed : {len(completed)}/{len(all_tasks)}")
        print(f"Execution waves : {len(waves)}")
        print(f"Total time      : {elapsed}s")
        print(f"Agent actions   : {len(trace.actions)} (full audit trail)")
        print("\nFinal output (Critic Agent validation):")
        print(f"  {final_output['final_result'][:300]}")

        return final_output


# ── RUN THE SYSTEM ────────────────────────────────────────────────
orchestrator = AgenticOrchestrator()

result = orchestrator.run(
    goal="Analyse customer feedback data and generate an actionable "
         "improvement report for the product team, prioritised by impact."
)


AGENTIC AI ORCHESTRATOR — STARTING
Goal: Analyse customer feedback data and generate an actionable improvement report for the product team, prioritised by impact.

--- PHASE 1: PLANNING ---
  [14:06:44.747] PlannerAgent | plan | confidence=0.92
  [14:06:44.747] PlannerAgent | plan | confidence=0.95

--- PHASE 2: SCHEDULING ---
  [14:06:44.747] SchedulerAgent | schedule | confidence=0.98
  [14:06:44.747] SchedulerAgent | allocate | confidence=0.96
  [14:06:44.747] SchedulerAgent | allocate | confidence=0.96
  [14:06:44.748] SchedulerAgent | allocate | confidence=0.96
  [14:06:44.748] SchedulerAgent | allocate | confidence=0.96

--- PHASE 3: EXECUTION ---

  Wave 0: Parallel execution (2 tasks)
    Executing: Requirements analysis [T1]
  [14:06:44.748] AnalysisAgent | execute | Task: T1 | confidence=0.88
  [14:06:44.848] AnalysisAgent | execute | Task: T1 | confidence=0.91
    Executing: Data retrieval [T2]
  [14:06:44.848] DataAgent | execute | Task: T2 | confidence=0.88
  [14:06:44.94

In [7]:
# ── Full explainability output ────────────────────────────────────
# This is the human-in-the-loop audit trail.
# Every agent decision, with reasoning, is logged here.
# Human operators can review, challenge, or override any action.

print("="*60)
print("FULL AGENT DECISION AUDIT TRAIL")
print("="*60)
print("(This is the explainability layer — every agent")
print(" action is logged with its reasoning so human")
print(" operators can understand and audit the system)\n")

for i, action in enumerate(trace.actions, 1):
    print(f"[{i:02d}] {action.timestamp} | {action.agent_name}")
    print(f"      Type      : {action.action_type}")
    if action.task_id:
        print(f"      Task      : {action.task_id}")
    print(f"      Reasoning : {action.reasoning}")
    print(f"      Confidence: {action.confidence:.2f}")
    print(f"      Output    : {action.output_data[:120]}")
    print()

trace.summary()
print("\n✅ Full audit trail printed")
print("Agentic AI that is robust, controllable, and explainable.")

FULL AGENT DECISION AUDIT TRAIL
(This is the explainability layer — every agent
 action is logged with its reasoning so human
 operators can understand and audit the system)

[01] 14:06:44.747 | PlannerAgent
      Type      : plan
      Reasoning : Goal requires multiple specialist agents. Decomposing by domain to enable parallel execution where dependencies allow.
      Confidence: 0.92
      Output    : Decomposing into subtask dependency graph

[02] 14:06:44.747 | PlannerAgent
      Type      : plan
      Reasoning : T1 and T2 have no dependencies so can run in parallel. T3 requires both to complete. T4 and T5 are sequential.
      Confidence: 0.95
      Output    : Created 5 tasks with dependency graph

[03] 14:06:44.747 | SchedulerAgent
      Type      : schedule
      Reasoning : Topological sort guarantees valid execution order. Tasks in same wave can run in parallel — maximising throughput while respecting dependencies.
      Confidence: 0.98
      Output    : Running topologic